In [ ]:
# Path setup — auto-finds sourceCode_PhiSpaceST/ and loads the `paths` dict.
import sys as _sys, os as _os
from pathlib import Path as _Path
_root = _Path.cwd().resolve()
while not (_root / 'config' / 'paths_template.yml').exists():
    if _root.parent == _root:
        raise FileNotFoundError('Run notebook from inside sourceCode_PhiSpaceST/.')
    _root = _root.parent
_sys.path.insert(0, str(_root / 'scripts' / '00_setup'))
from paths_loader import paths

In [ ]:
import scanpy as sc
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl

import cell2location

from matplotlib import rcParams
rcParams['pdf.fonttype'] = 42 # enables correct plotting of text for PDFs

Load trained model. (Consider retrain using filtered features.)

In [ ]:
results_folder = paths["phispace_data_root"] + "/"
# create paths and names to results folders for reference regression and cell2location models
ref_run_name = f'{results_folder}output/Case3/cell2location/reference_signatures'

adata_ref = sc.read(
    f'{results_folder}data/LungRef/AzimuthLung2.0_sce_0.1sub_selectFeat.h5ad'
)

In [ ]:
# prepare anndata for the regression model
cell2location.models.RegressionModel.setup_anndata(adata=adata_ref,
                        # 10X reaction / sample / batch
                        # batch_key='Sample',
                        # cell type, covariate used for constructing signatures
                        labels_key='ann_finest_level',
                        # multiplicative technical effects (platform, 3' vs 5', donor effect)
                        # categorical_covariate_keys=['Method']
                       )

In [ ]:
# create the regression model
from cell2location.models import RegressionModel
mod = RegressionModel(adata_ref)

# view anndata_setup as a sanity check
mod.view_anndata_setup()

In [ ]:
mod.train(max_epochs=250)

In [ ]:
# In this section, we export the estimated cell abundance (summary of the posterior distribution).
adata_ref = mod.export_posterior(
    adata_ref, sample_kwargs={'num_samples': 1000, 'batch_size': 2500}
)

In [ ]:
# Save model
mod.save(f"{ref_run_name}", overwrite=True)

# Save anndata object with results
adata_file = f"{ref_run_name}/sc.h5ad"
adata_ref.write(adata_file)